# 01 - Data Preparation

> Scaffold notebook — fill with data and code.

## Setup

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")
df = pd.read_csv(DATA_DIR / "matches.csv")
df.head(), len(df)


## Load data

In [ ]:
teams = pd.unique(df[["home_team", "away_team"]].values.ravel())
elo = {team: 1500 for team in teams}
elo


## Clean & normalize

In [ ]:
def update_elo(elo, home, away, home_goals, away_goals, k=20):
    if home_goals > away_goals:
        result = 1
    elif home_goals < away_goals:
        result = 0
    else:
        result = 0.5

    # expected score
    expected = 1 / (1 + 10 ** ((elo[away] - elo[home]) / 400))

    # update
    elo[home] += k * (result - expected)
    elo[away] -= k * (result - expected)

for _, r in df.iterrows():
    update_elo(elo, r["home_team"], r["away_team"], r["home_goals"], r["away_goals"])

pd.DataFrame({"team": list(elo.keys()), "elo": list(elo.values())})

df_elo = pd.DataFrame({"team": list(elo.keys()), "elo": list(elo.values())})
out_path = Path("../data/processed/elo.csv")
df_elo.to_csv(out_path, index=False)
print("Saved:", out_path)
